In [1]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT / "src"))
ART = ROOT / "artifacts"
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
from plotting import series_styles, new_figure

plt.rcParams.update({"figure.dpi": 110, "font.size": 9})

def es(v, d=4):
    """Numero con formato de Colombia: punto de miles y coma decimal."""
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return ""
    if isinstance(v, (int, np.integer)):
        return f"{int(v):,}".replace(",", ".")
    s = f"{float(v):,.{d}f}"
    return s.replace(",", "X").replace(".", ",").replace("X", ".")

def tabla(df, d=4, enteros=()):
    out = df.copy()
    for c in out.columns:
        if c in enteros:
            out[c] = out[c].map(lambda v: es(int(v)) if pd.notna(v) else "")
        elif pd.api.types.is_float_dtype(out[c]):
            out[c] = out[c].map(lambda v: es(v, d))
        elif pd.api.types.is_integer_dtype(out[c]):
            out[c] = out[c].map(es)
    display(out.style.hide(axis="index"))

def figura(ruta, ancho=None):
    display(Image(filename=str(ruta), width=ancho))

# 3. Protocolo anti-fuga y preprocesamiento

Este capítulo describe cómo se protegen los resultados contra la fuga de información y cómo se transforman las variables en ambos entornos. Es primordial dejarlo explícito, porque la fuga es la forma más común de que un modelo de CTR parezca mejor en el informe de lo que será en producción, y casi nunca produce un error visible: solo infla las métricas.

## 3.1 Partición temporal

El CTR tiene deriva diaria (sección 2.3), así que la partición es temporal y no aleatoria, tal como fija el enunciado: entrenamiento con los días 21 a 28, validación con el día 29 y prueba con el día 30. La validación se usa para elegir hiperparámetros y el umbral de decisión. La prueba se evalúa una sola vez por corrida, con el modelo y el umbral ya congelados.

| Partición | Días | Filas | Uso |
|---|---|---|---|
| Entrenamiento | 21 a 28 | 32.377.421 | Ajuste de codificadores, escaladores y modelos |
| Validación | 29 | 3.832.608 | Hiperparámetros y umbral de decisión |
| Prueba | 30 | 4.218.938 | Evaluación final, una vez por corrida |

## 3.2 Toda transformación se ajusta solo con entrenamiento

Los codificadores, el escalador y el umbral de categorías raras se ajustan con los días de entrenamiento de cada corrida, y luego se aplican sin reajustar a validación y prueba. Para que una violación de esta regla no pase en silencio, cada método `fit` verifica que no reciba filas con día posterior al último de entrenamiento y, si las recibe, lanza un `LeakageError`. El código de la guarda es el siguiente.

In [2]:
import inspect
import preprocessing
print(inspect.getsource(preprocessing.assert_is_train_only))

def assert_is_train_only(df: pd.DataFrame, max_day: int, day_col: str = "day") -> None:
    if day_col in df.columns and (df[day_col] > max_day).any():
        raise LeakageError(
            f"Se intento ajustar un encoder/escalador con dias > {max_day} "
            "(validacion o prueba). Revisa el pipeline de llamada."
        )



## 3.3 Agrupación de categorías raras

Las categorías que aparecen menos de 20 veces en entrenamiento se reemplazan por el token `__RARE__`, en las 12 columnas categóricas que usa el modelo: 5 de baja cardinalidad (`banner_pos`, `site_category`, `app_category`, `device_type`, `device_conn_type`) y 7 de alta (`site_id`, `site_domain`, `app_id`, `app_domain`, `device_id`, `device_ip`, `device_model`). Las columnas anonimizadas `C1` y `C14` a `C21` y la hora no forman parte del conjunto de variables del experimento principal; su aporte se mide en las ablaciones (capítulo 9). El umbral se calcula sobre las frecuencias de entrenamiento de cada corrida y se aplica igual a validación y prueba, donde cualquier categoría no vista en entrenamiento también cae en `__RARE__`.

Esto resuelve dos problemas a la vez. Las categorías vistas una vez, que en `device_ip` son el 49,4 % y en `device_id` el 62,9 % (sección 2.6), dejan de ocupar posiciones propias en el hashing, y las categorías nuevas del día 30 reciben un tratamiento coherente, en lugar de un valor arbitrario.

## 3.4 Estrategias de codificación

### Hashing trick

El hashing proyecta cada par variable-valor, por ejemplo `site_id=1fbe01fe`, a una de $d$ posiciones mediante una función hash $h$, y le asigna un signo $\xi \in \{-1, +1\}$ con una segunda función {cite}`weinberger2009feature`:

$$\phi_j(x) = \sum_{k \,:\, h(k) = j} \xi(k)\, x_k .$$

El signo hace que, en esperanza, las colisiones se cancelen en lugar de acumularse, de modo que el producto interno entre vectores hasheados es un estimador insesgado del original. Con $d = 2^{12} = 4.096$ posiciones y 13 pares por fila (12 categóricas más el cruce), la probabilidad de que dos pares de una misma fila colisionen es $1/4.096$, y el número esperado de colisiones por fila es $\binom{13}{2}/4.096 = 78/4.096 \approx 0{,}019$: en promedio, menos del 2 % de las filas tiene alguna colisión. Las colisiones entre categorías de filas distintas son mucho más frecuentes, porque millones de valores de `device_ip` comparten 4.096 posiciones, pero la agrupación de raras ya manda la mayoría de esos valores a un solo token. La dimensión del hashing es un factor de las ablaciones (capítulo 9), con los niveles $2^{12}$, $2^{14}$ y $2^{16}$ del enunciado, más $2^{10}$.

### Codificación por frecuencia

Cada categoría se reemplaza por su frecuencia relativa en entrenamiento, $f(c) = n_c / N_{\text{train}}$. Es la codificación más barata en memoria, una columna por variable, pero pierde la identidad de la categoría: dos sitios con la misma frecuencia reciben el mismo valor aunque su CTR sea muy distinto.

### Target encoding fuera de pliegue

Cada categoría se reemplaza por su CTR suavizado hacia la media global $\bar y$ {cite}`micci2001preprocessing`:

$$\mathrm{TE}(c) = \frac{n_c\, \bar y_c + m\, \bar y}{n_c + m}, \qquad m = 20 .$$

Con pocas observaciones, $n_c \ll m$ y el valor se acerca a la tasa base, lo que evita que una categoría con dos impresiones y un clic reciba un CTR de 50 %. El riesgo es que, si se calcula sobre las mismas filas que luego se usan para entrenar, cada fila "ve" su propia etiqueta a través del codificador, y el modelo aprende una relación que no existe en datos nuevos. Por eso el enunciado exige validación cruzada interna: el entrenamiento se divide en 5 pliegues y cada fila recibe el valor calculado solo con los otros 4. Validación y prueba reciben el mapa ajustado con todo el entrenamiento. El enunciado limita esta codificación a scikit-learn, y así se implementa.

### Escalado y cruce de variables

El bloque denso de frecuencia y target encoding se estandariza con un `StandardScaler` ajustado en entrenamiento. El cruce `site_id` × `device_type` concatena ambas variables en una nueva categórica que entra al hashing, y se evalúa como ablación.

La transformación logarítmica para numéricas de cola pesada que menciona el enunciado no se aplica, porque Avazu no tiene numéricas continuas: las columnas `C1` y `C14` a `C21` son códigos anonimizados, y la hora es acotada y cíclica (sección 2.4). Aplicar un logaritmo a un identificador no tiene sentido estadístico.

## 3.5 Preprocesamiento en cada entorno

El enunciado pide la misma lógica de preprocesamiento en ambos entornos, y a la vez fija el pipeline de PySpark con `FeatureHasher`, `StringIndexer` solo para baja cardinalidad, `VectorAssembler` y `StandardScaler`. Se respetó la composición literal de la sección de PySpark, lo que deja diferencias que conviene declarar.

| Paso | scikit-learn | PySpark |
|---|---|---|
| Categorías raras | `__RARE__`, umbral 20 en train | No (Spark ML no trae un agrupador nativo) |
| Alta cardinalidad | Hashing $2^{12}$ con signo | `FeatureHasher` $2^{12}$ (LR); $2^{9}$ en MLP |
| Baja cardinalidad | Dentro del hashing | `StringIndexer` con `handleInvalid="keep"`; el índice entra como valor numérico, sin one-hot |
| Columnas `C*` y hora | No incluidas | No incluidas |
| Frecuencia y target encoding | Sí, en las 7 de alta cardinalidad | No (target encoding limitado a sklearn por el enunciado) |
| Cruce `site_id` × `device_type` | Sí | No |
| Escalado | `StandardScaler` sobre el bloque denso | `StandardScaler(withMean=False)` sobre el vector ensamblado |

En consecuencia, la comparación entre entornos del capítulo 6 no aísla el efecto del entorno: compara "entorno más preprocesamiento disponible de forma nativa". Hay un detalle del pipeline de Spark que pesa en esa comparación: `StringIndexer` convierte cada categoría en un índice entero, y sin un `OneHotEncoder` posterior ese índice entra a la regresión logística y al MLP como si fuera una magnitud, con un orden que no tiene sentido. El enunciado no incluye el `OneHotEncoder` en el pipeline, y se respetó su composición. El capítulo 5 detalla además los ajustes que impuso la memoria en el pipeline de Spark.

## 3.6 Tamaños efectivos de las muestras

Durante la auditoría de los resultados se encontró que DuckDB aplica la cláusula `USING SAMPLE` antes del `WHERE`. Las consultas de muestreo de `sklearn_pipeline.py` y del escalamiento fuerte y débil tomaban $n$ filas del dataset completo y después filtraban por día, de modo que el tamaño efectivo es $n$ multiplicado por la fracción de filas de esos días. No hay fuga, porque el filtro temporal se sigue aplicando sobre cada fila, pero los tamaños nominales no coinciden con los reales. La tabla reporta los tamaños efectivos, que son los que se usan en todo el libro.

In [3]:
t = json.loads((ART / "benchmarking" / "tamanos_efectivos.json").read_text(encoding="utf-8"))
filas = [("scikit-learn, tier nominal " + es(int(k)), v) for k, v in t["sklearn_tiers_nominal_a_efectivo"].items()]
filas += [("scikit-learn, validación (aprox.)", t["sklearn_validacion_efectiva_aprox"]),
          ("scikit-learn, prueba (exacto)", t["sklearn_prueba_efectiva_exacta"]),
          ("Escalamiento fuerte (aprox.)", t["strong_scaling_efectivo_aprox"])]
filas += [(f"Escalamiento débil, {p} núcleo(s) (aprox.)", v) for p, v in t["weak_scaling_efectivo_aprox"].items()]
df = pd.DataFrame(filas, columns=["muestra", "filas efectivas"])
tabla(df, enteros=("filas efectivas",))
print(f"Fracción de filas de entrenamiento en el dataset: {es(t['fraccion_train'], 4)}")

muestra,filas efectivas
"scikit-learn, tier nominal 100.000",80.085
"scikit-learn, tier nominal 1.000.000",800.847
"scikit-learn, tier nominal 2.000.000",1.601.694
"scikit-learn, validación (aprox.)",189.597
"scikit-learn, prueba (exacto)",202.511
Escalamiento fuerte (aprox.),1.601.694
"Escalamiento débil, 1 núcleo(s) (aprox.)",800.847
"Escalamiento débil, 2 núcleo(s) (aprox.)",1.601.694
"Escalamiento débil, 4 núcleo(s) (aprox.)",3.203.388
"Escalamiento débil, 8 núcleo(s) (aprox.)",6.406.777


Fracción de filas de entrenamiento en el dataset: 0,8008


**Interpretación.** Los días 21 a 28 son el 80,08 % del dataset, así que los tiers nominales de 100 mil, 1 millón y 2 millones de filas de scikit-learn tienen en realidad 80.085, 800.847 y 1.601.694 filas de entrenamiento. La validación tiene unas 189.597 filas y la prueba exactamente 202.511, que es el número de etiquetas guardadas con las predicciones. Los scripts que se escribieron después de detectar el comportamiento (interpretabilidad, ablaciones y benchmarking por etapas) usan una subconsulta que filtra primero y muestrea después, y obtienen exactamente el número de filas pedido.

Esto significa que las curvas de aprendizaje cubren un rango de 20 veces en volumen, de 80 mil a 1,6 millones de filas, en lugar del rango de 100 veces que habría dado el diseño original con 10 millones. El enunciado pide tamaños de 100 mil, 1 millón y 10 millones "hasta donde permita la memoria", y el tope de 2 millones nominales responde al tiempo de cómputo medido con el MLP sobre $2^{14}$ columnas (capítulo 4). Una prueba de 202.511 filas, con unos 34 mil positivos, da un error estándar del AUC cercano a 0,0016 según la aproximación de Hanley y McNeil {cite}`hanley1982meaning`, suficiente para las comparaciones del capítulo 6.

Así las cosas, se decide conservar los resultados ya calculados, porque son válidos y libres de fuga, y reportar siempre los tamaños efectivos. El muestreo de los scripts nuevos fija además un solo hilo en DuckDB, porque con varios hilos el muestreo por reservorio no es determinista; con uno solo, la misma semilla devuelve exactamente las mismas filas, lo que se verificó comparando los identificadores de dos ejecuciones.

## 3.7 Lista de verificación anti-fuga

| Control | Cómo se garantiza |
|---|---|
| Partición temporal fija | Filtro por día en todas las consultas; la prueba es el día 30 |
| Codificadores y escalador solo con entrenamiento | `assert_is_train_only` en cada `fit`, que lanza `LeakageError` |
| Target encoding sin ver la propia etiqueta | Cinco pliegues fuera de pliegue en entrenamiento |
| Hiperparámetros elegidos sin la prueba | Optuna y la selección de Spark evalúan contra el día 29; `TrainValidationSplit` no se usa porque haría una división aleatoria interna |
| Umbral elegido sin la prueba | Máximo F1 en validación, congelado antes de predecir el día 30 |
| Prueba evaluada una vez por corrida | Cada corrida guarda sus predicciones de prueba una sola vez |